# Image Clustering dengan K-Means

Notebook ini digunakan untuk melakukan clustering pada data gambar menggunakan K-Means.

Tahapan analisis:
1. Import library
2. Load gambar dan label
3. Ekstraksi fitur RGB dan HSV
4. Standardisasi fitur
5. Elbow Method
6. Silhouette Score
7. K-Means Clustering
8. PCA visualization
9. Confusion Matrix
10. Menyimpan hasil clustering

Untuk Confusion Matrix, label asli diperoleh dari nama folder gambar.

## 1. Import Library

Library digunakan untuk membaca gambar, melakukan ekstraksi fitur,
clustering, evaluasi silhouette, PCA, dan confusion matrix.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment

## 2. Konfigurasi

Atur lokasi folder gambar dan parameter clustering pada bagian berikut.
Jika `SELECTED_K = None`, jumlah cluster dengan Silhouette Score terbaik
akan digunakan secara otomatis.

In [ ]:
IMAGE_FOLDER = "data/images"

K_MIN = 2
K_MAX = 8
SELECTED_K = None

RANDOM_STATE = 42
IMAGE_SIZE = (128, 128)
HIST_BINS = 16

## 3. Load Gambar dan Label

Setiap gambar dibaca dari folder. Jika gambar berada di dalam subfolder,
nama subfolder digunakan sebagai label asli.

Contoh:
`data/images/kucing/gambar1.jpg` memiliki label `kucing`.

In [ ]:
def get_image_paths_and_labels(folder):
    extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
    paths = []

    for ext in extensions:
        paths.extend(glob.glob(os.path.join(folder, "**", ext), recursive=True))
        paths.extend(glob.glob(os.path.join(folder, "**", ext.upper()), recursive=True))

    paths = sorted(list(set(paths)))

    labels = []
    for path in paths:
        relative = os.path.relpath(path, folder)
        parts = relative.split(os.sep)

        if len(parts) > 1:
            labels.append(parts[0])
        else:
            labels.append(None)

    return paths, labels


paths, labels = get_image_paths_and_labels(IMAGE_FOLDER)

if len(paths) == 0:
    raise FileNotFoundError(f"Tidak ada gambar ditemukan di: {IMAGE_FOLDER}")

print(f"Jumlah gambar: {len(paths)}")

## 4. Ekstraksi Fitur Gambar

Fitur yang digunakan berasal dari karakteristik warna RGB dan HSV,
meliputi mean, standard deviation, dan histogram warna.

In [ ]:
def extract_image_features(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMAGE_SIZE)

    rgb = np.asarray(img, dtype=np.float32) / 255.0

    rgb_mean = rgb.mean(axis=(0, 1))
    rgb_std = rgb.std(axis=(0, 1))

    rgb_hist = []
    for c in range(3):
        hist, _ = np.histogram(
            rgb[:, :, c],
            bins=HIST_BINS,
            range=(0, 1),
            density=True
        )
        hist = hist / (hist.sum() + 1e-12)
        rgb_hist.extend(hist)

    hsv_img = Image.fromarray((rgb * 255).astype(np.uint8)).convert("HSV")
    hsv = np.asarray(hsv_img, dtype=np.float32) / 255.0

    hsv_mean = hsv.mean(axis=(0, 1))
    hsv_std = hsv.std(axis=(0, 1))

    hsv_hist = []
    for c in range(3):
        hist, _ = np.histogram(
            hsv[:, :, c],
            bins=HIST_BINS,
            range=(0, 1),
            density=True
        )
        hist = hist / (hist.sum() + 1e-12)
        hsv_hist.extend(hist)

    return np.concatenate([
        rgb_mean,
        rgb_std,
        rgb_hist,
        hsv_mean,
        hsv_std,
        hsv_hist
    ])


features = []
valid_paths = []
valid_labels = []

for i, path in enumerate(paths):
    try:
        features.append(extract_image_features(path))
        valid_paths.append(path)
        valid_labels.append(labels[i])
    except Exception as e:
        print(f"Gagal membaca {path}: {e}")

X = np.array(features)

X = SimpleImputer(strategy="median").fit_transform(X)
X_scaled = StandardScaler().fit_transform(X)

print("Shape data fitur:", X_scaled.shape)

## 5. Elbow Method

Elbow Method digunakan untuk melihat perubahan inertia pada beberapa
nilai K. Nilai K dapat dipertimbangkan dari titik ketika penurunan inertia
mulai melandai.

In [ ]:
max_k = min(K_MAX, len(X_scaled) - 1)

if max_k < K_MIN:
    raise ValueError("Jumlah data terlalu sedikit.")

k_values = range(K_MIN, max_k + 1)
inertias = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method - Image Clustering")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Silhouette Score

Silhouette Score digunakan untuk mengevaluasi kualitas pemisahan cluster.
Nilai yang lebih tinggi menunjukkan struktur cluster yang lebih baik.

In [ ]:
silhouette_results = {}

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    cluster_labels = model.fit_predict(X_scaled)
    silhouette_results[k] = silhouette_score(X_scaled, cluster_labels)

print("Silhouette Score:")
for k, score in silhouette_results.items():
    print(f"K={k}: {score:.4f}")

best_k = max(silhouette_results, key=silhouette_results.get)

if SELECTED_K is None:
    SELECTED_K = best_k

print(f"\nK yang digunakan: {SELECTED_K}")

## 7. K-Means Clustering

K-Means dijalankan menggunakan K yang telah dipilih.

In [ ]:
kmeans = KMeans(
    n_clusters=SELECTED_K,
    random_state=RANDOM_STATE,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

final_silhouette = silhouette_score(X_scaled, clusters)

print(f"Silhouette Score final: {final_silhouette:.4f}")

if final_silhouette >= 0.70:
    print("Interpretasi: struktur cluster sangat baik.")
elif final_silhouette >= 0.50:
    print("Interpretasi: struktur cluster cukup baik.")
elif final_silhouette >= 0.25:
    print("Interpretasi: struktur cluster masih lemah/sedang.")
else:
    print("Interpretasi: struktur cluster kurang baik.")

## 8. PCA Visualization

PCA digunakan untuk mereduksi fitur menjadi dua komponen utama
agar hasil clustering dapat divisualisasikan dalam bidang dua dimensi.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters,
    cmap="tab10",
    s=50
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA - Image Clusters")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 9. Jumlah Data Setiap Cluster

In [ ]:
unique, counts = np.unique(clusters, return_counts=True)

for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} gambar")

## 10. Confusion Matrix

Karena K-Means menghasilkan nomor cluster dan bukan nama kelas,
cluster dipetakan terlebih dahulu ke label asli menggunakan
Hungarian Algorithm. Setelah itu confusion matrix dan accuracy
dapat dihitung.

In [ ]:
def best_cluster_label_mapping(y_true, y_cluster):
    true_labels = np.unique(y_true)
    cluster_labels = np.unique(y_cluster)

    true_to_int = {label: i for i, label in enumerate(true_labels)}
    y_true_int = np.array([true_to_int[x] for x in y_true])

    n = max(len(true_labels), len(cluster_labels))
    contingency = np.zeros((n, n), dtype=int)

    for t, c in zip(y_true_int, y_cluster):
        contingency[t, c] += 1

    row_ind, col_ind = linear_sum_assignment(-contingency)

    mapping = {c: r for r, c in zip(row_ind, col_ind)}

    for c in cluster_labels:
        if c not in mapping:
            mapping[c] = 0

    y_pred_mapped = np.array([mapping[c] for c in y_cluster])

    return true_labels, y_true_int, y_pred_mapped


valid = [x is not None for x in valid_labels]

if all(valid):
    true_labels, y_true_int, y_pred_mapped = best_cluster_label_mapping(
        np.array(valid_labels),
        np.array(clusters)
    )

    cm = confusion_matrix(
        y_true_int,
        y_pred_mapped,
        labels=np.arange(len(true_labels))
    )

    accuracy = accuracy_score(y_true_int, y_pred_mapped)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=true_labels,
        yticklabels=true_labels
    )
    plt.xlabel("Predicted Cluster")
    plt.ylabel("Actual Label")
    plt.title(f"Confusion Matrix - Image Clustering\nAccuracy = {accuracy:.4f}")
    plt.tight_layout()
    plt.show()

    print("Confusion Matrix:")
    print(pd.DataFrame(cm, index=true_labels, columns=true_labels))
    print(f"Accuracy: {accuracy:.4f}")
else:
    print("Confusion Matrix tidak dapat dibuat karena label asli tidak tersedia.")

## Helper Interpretasi Otomatis

Cell berikut membaca hasil clustering, silhouette score, PCA, dan confusion matrix lalu memberikan interpretasi otomatis.

In [ ]:
# Helper Interpretasi Hasil Image Clustering

print("=" * 70)
print("HELPER INTERPRETASI IMAGE CLUSTERING")
print("=" * 70)

# 1. Silhouette
print("\n1. INTERPRETASI SILHOUETTE SCORE")
print(f"Silhouette Score = {final_silhouette:.4f}")

if final_silhouette >= 0.70:
    print("Hasil: sangat baik.")
    print("Data memiliki pemisahan cluster yang sangat jelas.")
elif final_silhouette >= 0.50:
    print("Hasil: cukup baik.")
    print("Sebagian besar gambar memiliki pemisahan cluster yang cukup jelas.")
elif final_silhouette >= 0.25:
    print("Hasil: sedang/lemah.")
    print("Beberapa gambar masih memiliki kedekatan dengan cluster lain.")
else:
    print("Hasil: kurang baik.")
    print("Pemisahan antar cluster masih lemah.")

# 2. Cluster size
print("\n2. DISTRIBUSI CLUSTER")
cluster_counts = pd.Series(clusters).value_counts().sort_index()

for cluster_id, count in cluster_counts.items():
    percentage = count / len(clusters) * 100
    print(f"Cluster {cluster_id}: {count} gambar ({percentage:.2f}%)")

largest_cluster = cluster_counts.idxmax()
smallest_cluster = cluster_counts.idxmin()

print(
    f"Cluster terbesar adalah Cluster {largest_cluster} "
    f"dengan {cluster_counts[largest_cluster]} gambar."
)
print(
    f"Cluster terkecil adalah Cluster {smallest_cluster} "
    f"dengan {cluster_counts[smallest_cluster]} gambar."
)

# 3. Confusion Matrix
print("\n3. INTERPRETASI CONFUSION MATRIX")

if all(valid):
    print(f"Accuracy setelah mapping cluster-label = {accuracy:.4f}")

    if accuracy >= 0.90:
        print("Hasil: sangat baik. Sebagian besar gambar berhasil")
        print("dikelompokkan sesuai dengan label aslinya.")
    elif accuracy >= 0.75:
        print("Hasil: baik. Mayoritas gambar sudah berada pada")
        print("cluster yang sesuai dengan label aslinya.")
    elif accuracy >= 0.60:
        print("Hasil: cukup. Masih terdapat beberapa kesalahan")
        print("pengelompokan antar kelas.")
    else:
        print("Hasil: kurang baik. Banyak gambar masih tertukar")
        print("antar kelompok.")

    print("\nRingkasan kecocokan label:")
    for i, label in enumerate(true_labels):
        row_total = cm[i].sum()
        correct = cm[i, i]
        pct = correct / row_total * 100 if row_total else 0
        print(
            f"- {label}: {correct}/{row_total} "
            f"benar ({pct:.2f}%)"
        )
else:
    print("Confusion Matrix tidak dapat diinterpretasikan karena")
    print("label asli tidak tersedia.")

# 4. PCA
print("\n4. INTERPRETASI PCA")
explained = pca.explained_variance_ratio_
print(
    f"PC1 menjelaskan {explained[0] * 100:.2f}% variasi data."
)
print(
    f"PC2 menjelaskan {explained[1] * 100:.2f}% variasi data."
)
print(
    f"Total PC1 + PC2 = {explained.sum() * 100:.2f}% variasi data."
)

print("\nKESIMPULAN IMAGE CLUSTERING")
print(
    f"Data gambar dibagi menjadi {SELECTED_K} cluster dengan "
    f"Silhouette Score {final_silhouette:.4f}."
)